In [1]:
from typing import List, Sequence
from autogen_agentchat.teams import SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.ui import Console
from autogen_agentchat.agents import UserProxyAgent, AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
import os 
import sys
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
load_dotenv()

True

This system uses three specialized agents:

Planning Agent: The strategic coordinator that breaks down complex tasks into manageable subtasks.

Web Search Agent: An information retrieval specialist that interfaces with the web_search_tool.

Data Analyst Agent: An agent specialist in performing calculations equipped with percentage_change_tool.

The tools search_web_tool and percentage_change_tool are external tools that the agents can use to perform their tasks.

In [ ]:
model_client = OpenAIChatCompletionClient(
    model='gpt-3.5-turbo'
)

In [3]:
#mock tool
def web_search_tool(query:str) -> str:
    if "2007-2008" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."

In [5]:
def percentage_change_tool(start:float, end: float)-> float:
    return ((end - start) / start) * 100

Now specialized agents are created using the AssistantAgent class. It is important to note that the agents’ name and description attributes are used by the model to determine the next speaker, so it is recommended to provide meaningful names and descriptions.

In [6]:
planning_agent = AssistantAgent(
    "PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Perform calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)